# 9. Classification
In this notebook, we explore whether machine learning can be used to classify samples based on their microbiome composition. The aim of this analysis is to assess whether microbial profiles contain enough information to predict selected metadata variables, such as feeding pattern, delivery mode, treatment exposure, or geographic origin. This work is exploratory and focuses on understanding the potential of microbiome-based classification rather than producing final results.

The analysis starts with basic preparation of the microbiome data and metadata. Using the processed data, a supervised classifier is trained to predict the selected metadata category from microbiome composition. The dataset is split into training and test sets so that model performance can be evaluated on unseen samples. Model accuracy and feature importance are examined to obtain an initial impression of how well different groups can be distinguished and which taxa contribute to the predictions.

This notebook has some limitations. In particular, the classification pipeline does not correct for host identity. Because multiple samples from the same individual may be included, samples from one host can appear in both the training and test sets. As a result, the classifier may partly learn host-specific microbiome patterns rather than differences related to the metadata variable of interest. Addressing this issue would require group-aware data splitting or alternative modeling approaches that are not available within the current QIIME workflow and would need to be implemented using other Python machine learning packages. Also, while basic model evaluation is performed, more detailed checks for overfitting or underfitting are not yet included. 

For these reasons, **the results from this notebook are not used in the main report or results sections.** Instead, this notebook serves as an exploratory analysis and a starting point for further work. With several improvements, including correction for host identity, this classification approach could used for further analyses.

### Notebook Structure

**0.** Setup  
**1.** Classification Pipeline     
**2.** Classification Results  


## 0. Setup

In [1]:
# Import all necessary packages
import pandas as pd
import numpy as np
import glob
import re
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
import seaborn as sns
import os
import qiime2 as q2
from qiime2 import Visualization
from biom import load_table
from ipywidgets import Dropdown, VBox, HBox
from IPython.display import Image, display

%matplotlib inline

In [2]:
# The working directory should normally default to the 'scripts' folder. 
# If it doesn't, set it manually using the command below.
# os.chdir("/home/jovyan/MicrobiomeAnalysis_TummyTribe/scripts")  # Adjust this path to match your folder structure.

# Verify that your working directory is the 'scripts' folder inside the main project directory (.../MicrobiomeAnalysis_TummyTribe/scripts)
cwd = os.getcwd()
if not cwd.endswith("MicrobiomeAnalysis_TummyTribe/scripts"):
    print("WARNING: The working directory is not set to the 'scripts' folder inside 'MicrobiomeAnalysis_TummyTribe'!")
    print("Current working directory:", cwd)
    print("Please set it manually using os.chdir().")
else:
    print(f"Working directory is correctly set to the 'scripts' folder (\"{cwd}\").")

Working directory is correctly set to the 'scripts' folder ("/home/jovyan/MicrobiomeAnalysis_TummyTribe/scripts").


In [3]:
# Data directories
raw_data_dir = "../data/raw"
processed_data_dir = "../data/processed"
meta_data_dir = "../data/processed/metadata"
raw_data_dir = "../data/raw"
denoised_data_dir = "../data/processed/denoising"
taxonomy_data_dir = "../data/processed/taxonomy"
diversity_data_dir = "../data/processed/diversity"
phylogeny_data_dir = "../data/processed/phylogeny"
classification_dir = "../data/processed/classification"

In [4]:
%%bash -s "$classification_dir"
mkdir -p "$1"

### Set metrics to work with

The interactive widgets below allow you to configure how the classification is performed. First, you select a metadata variable that is used to analyze. Examples include geographic location, delivery mode, feeding pattern, or treatment exposure. Then, you can choose the taxonomic level at which the microbiome will be analyzed, for example phylum, family, or genus. This determines how detailed the microbial classification will be in the downstream analysis.



In [6]:
# 1. Select metadata column for classification
categorical_cols = ["geo_location_name", "delivery_mode", "sex", "diet_milk", "treatment_exposure"]

meta_selector = Dropdown(
    options=sorted(categorical_cols),
    value="diet_milk",
    description="Metadata:",
)

# 2. Select taxonomic level (all lowercase)
tax_levels = ["phylum", "class", "order", "family", "genus", "species"]

tax_selector = Dropdown(
    options=tax_levels,
    value="genus",
    description="Tax level:",
)

# 3. Select classifier
classifier_options = [
    'RandomForestClassifier',
    'ExtraTreesClassifier',
    'GradientBoostingClassifier',
    'AdaBoostClassifier[DecisionTree]',
    'AdaBoostClassifier[ExtraTrees]',
    'KNeighborsClassifier',
    'LinearSVC',
    'SVC'
]

classifier_selector = Dropdown(
    options=classifier_options,
    value='RandomForestClassifier',
    description="Classifier:",
)

VBox([
    meta_selector,
    tax_selector,
    classifier_selector,
])

In [7]:
# Read user selections
column_to_investigate = meta_selector.value
selected_level = tax_selector.value
classifier_choice = classifier_selector.value

## 1. Classification Pipeline

In this step, we prepare the feature table for classification by selecting a specific taxonomic level and organizing the output structure for the current analysis. Microbiome feature tables generated by DADA2 contain individual sequence variants, each annotated with a full taxonomic lineage. Since the goal of this analysis is to build a classifier at a biologically interpretable level, we restrict the feature table to a single taxonomic rank, such as genus or family.In this step, we prepare the output folder for this run and filter the feature table to the selected taxonomic level. The filtered table becomes the starting point for all of the next steps.

In [8]:
# Map taxonomy level to QIIME taxonomy prefix
prefix_map = {
    "phylum": "p__",
    "class": "c__",
    "order": "o__",
    "family": "f__",
    "genus": "g__",
    "species": "s__",
}
prefix = prefix_map[selected_level]

# Clean classifier name so it is safe for folder names
classifier_clean = classifier_choice.replace("[", "_").replace("]", "").replace(" ", "_")

# Create output directory for this classification run
out_dir = f"{classification_dir}/results-{column_to_investigate}-{selected_level}-{classifier_clean}"
! mkdir -p "{out_dir}"

# Filter the feature table to include only features assigned at the selected taxonomy level
! qiime taxa filter-table \
  --i-table "{denoised_data_dir}/dada2_table.qza" \
  --i-taxonomy "{taxonomy_data_dir}/taxonomy.qza" \
  --p-include "{prefix}" \
  --p-exclude "{prefix};" \
  --o-filtered-table "{out_dir}/{selected_level}_only_table.qza"

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureTable[Frequency] to: ../data/processed/classification/results-diet_milk-genus-RandomForestClassifier/genus_only_table.qza


After restricting the table to the selected taxonomic level, the next step is to collapse features so that all sequence variants assigned to the same taxon are combined into a single feature. Even after filtering, multiple ASVs may still map to the same genus or family, which would lead to redundant predictors in the classifier. Collapsing the table ensures that each feature corresponds to exactly one taxonomic unit at the chosen rank, with counts summed across all underlying ASVs. This results in a more compact and interpretable feature matrix that is better suited for downstream classification.

In [9]:
# Map taxonomy level to QIIME collapse level
qiime_level_map = {
    "phylum": 2,
    "class": 3,
    "order": 4,
    "family": 5,
    "genus": 6,
    "species": 7,
}
collapse_level = qiime_level_map[selected_level]

# Define input and output paths for collapsing
input_table = f"{out_dir}/{selected_level}_only_table.qza"
collapsed_out = f"{out_dir}/collapsed_table.qza"

print("Collapsing table at level:", selected_level)

# Collapse features so each taxon becomes a single feature
! qiime taxa collapse \
  --i-table "{input_table}" \
  --i-taxonomy "{taxonomy_data_dir}/taxonomy.qza" \
  --p-level {collapse_level} \
  --o-collapsed-table "{collapsed_out}"

Collapsing table at level: genus
/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureTable[Frequency] to: ../data/processed/classification/results-diet_milk-genus-RandomForestClassifier/collapsed_table.qza


Before training the classifier, the metadata table is cleaned to ensure that all samples included in the analysis have a valid value for the target variable. Since the classifier is supervised, each sample must have an associated label corresponding to the metadata column being predicted. The metadata file is loaded, sample identifiers are set as the index, and samples with missing values in the target column are removed. The cleaned metadata table is then saved to the output directory.

In [10]:
# Load merged metadata
m_raw = pd.read_csv(f"{meta_data_dir}/metadata_merged.tsv", sep="\t")

# Set sample ID as index (required by QIIME)
m_raw = m_raw.set_index("id")

# Remove samples without a value for the target variable
m = m_raw.dropna(subset=[column_to_investigate])

# Save cleaned metadata
clean_meta_path = f"{out_dir}/metadata_cleaned_{column_to_investigate}.tsv"
m.to_csv(clean_meta_path, sep="\t")

print("Saved cleaned metadata to:", clean_meta_path)
print("Number of samples:", len(m))

Saved cleaned metadata to: ../data/processed/classification/results-diet_milk-genus-RandomForestClassifier/metadata_cleaned_diet_milk.tsv
Number of samples: 270


After cleaning the metadata, the collapsed feature table is filtered to ensure that it contains exactly the same set of samples. This step aligns the microbiome feature matrix with the metadata labels used for classification, preventing inconsistencies during model training.

In [11]:
# Filter feature table to match cleaned metadata
! qiime feature-table filter-samples \
  --i-table "{out_dir}/collapsed_table.qza" \
  --m-metadata-file "{clean_meta_path}" \
  --o-filtered-table "{out_dir}/collapsed_table_filtered.qza"

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureTable[Frequency] to: ../data/processed/classification/results-diet_milk-genus-RandomForestClassifier/collapsed_table_filtered.qza


With both the feature table and metadata prepared, we can now train a supervised classifier using QIIME’s sample-classifier framework. The model is trained to predict the selected metadata variable from the taxonomic composition of each sample. A fixed random seed is used to ensure reproducibility of the train–test split, with 20% of samples held out for evaluation. All results from this classification run, including accuracy metrics and feature importance estimates, are stored in a dedicated output directory.

In [12]:
# Output directory for classifier results
qiime_clf_dir = f"{classification_dir}/qiime-classifier-{column_to_investigate}-{selected_level}-{classifier_clean}"

cmd = f"""
if [ -d "{qiime_clf_dir}" ]; then
    rm -r "{qiime_clf_dir}"
fi

qiime sample-classifier classify-samples \
  --i-table "{out_dir}/collapsed_table_filtered.qza" \
  --m-metadata-file "{clean_meta_path}" \
  --m-metadata-column "{column_to_investigate}" \
  --p-test-size 0.2 \
  --p-estimator "{classifier_choice}" \
  --p-random-state 15 \
  --p-n-jobs 3 \
  --output-dir "{qiime_clf_dir}"
"""

print(cmd)
!bash -c "{cmd}"


if [ -d "../data/processed/classification/qiime-classifier-diet_milk-genus-RandomForestClassifier" ]; then
    rm -r "../data/processed/classification/qiime-classifier-diet_milk-genus-RandomForestClassifier"
fi

qiime sample-classifier classify-samples   --i-table "../data/processed/classification/results-diet_milk-genus-RandomForestClassifier/collapsed_table_filtered.qza"   --m-metadata-file "../data/processed/classification/results-diet_milk-genus-RandomForestClassifier/metadata_cleaned_diet_milk.tsv"   --m-metadata-column "diet_milk"   --p-test-size 0.2   --p-estimator "RandomForestClassifier"   --p-random-state 15   --p-n-jobs 3   --output-dir "../data/processed/classification/qiime-classifier-diet_milk-genus-RandomForestClassifier"

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Ref

After classification, we load the accuracy visualization generated by QIIME. This allows us to inspect overall model performance.

## 1.2 Classification Results

After training the classifier, we first inspect the overall predictive performance on the held-out test set. QIIME automatically computes accuracy and related metrics and stores them as a visualization file. Loading this visualization provides a quick overview of how well the microbiome-based model can predict the selected metadata variable.

In [13]:
viz_path = f"{qiime_clf_dir}/accuracy_results.qzv"
Visualization.load(viz_path)

<visualization: Visualization uuid: ff3573c5-09f2-4a19-a5ca-72ed96a761f0>

To better understand how the classifier did at the level of individual samples, we next examine the true labels, predicted labels, and predicted class probabilities for the test set. These outputs allow us to look at misclassified samples and understand the model’s confidence for each prediction.

In [14]:
cmd = f"""
qiime metadata tabulate \
  --m-input-file "{qiime_clf_dir}/test_targets.qza" \
  --m-input-file "{qiime_clf_dir}/predictions.qza" \
  --m-input-file "{qiime_clf_dir}/probabilities.qza" \
  --o-visualization "{qiime_clf_dir}/test_predprob.qzv"
"""

print(cmd)
!bash -c "{cmd}"


qiime metadata tabulate   --m-input-file "../data/processed/classification/qiime-classifier-diet_milk-genus-RandomForestClassifier/test_targets.qza"   --m-input-file "../data/processed/classification/qiime-classifier-diet_milk-genus-RandomForestClassifier/predictions.qza"   --m-input-file "../data/processed/classification/qiime-classifier-diet_milk-genus-RandomForestClassifier/probabilities.qza"   --o-visualization "../data/processed/classification/qiime-classifier-diet_milk-genus-RandomForestClassifier/test_predprob.qzv"

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/opt/conda/lib/python3.10/site-packages/q2_sample_classifier/_transformer.py:76: FutureWarning: errors='ignore' is deprecated and will raise i

The resulting table is loaded below. Each row corresponds to a test sample and includes the true class label, the predicted label, and the associated prediction probabilities.

In [15]:
# Display predictions and probabilities for test samples
viz_path = f"{qiime_clf_dir}/test_predprob.qzv"
Visualization.load(viz_path)

<visualization: Visualization uuid: c1a87162-c2f2-4704-a66f-11d6cbaf4861>

To identify which taxa contribute most strongly to the classification, we examine the feature importance scores produced by the classifier. These scores reflect how informative each taxon is for distinguishing between classes in the trained model. 

In [16]:
cmd = f"""
qiime metadata tabulate \
  --m-input-file "{qiime_clf_dir}/feature_importance.qza" \
  --o-visualization "{qiime_clf_dir}/feature_importance.qzv"
"""

print(cmd)
!bash -c "{cmd}"


qiime metadata tabulate   --m-input-file "../data/processed/classification/qiime-classifier-diet_milk-genus-RandomForestClassifier/feature_importance.qza"   --o-visualization "../data/processed/classification/qiime-classifier-diet_milk-genus-RandomForestClassifier/feature_importance.qzv"

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/classification/qiime-classifier-diet_milk-genus-RandomForestClassifier/feature_importance.qzv


This step loads two key visualizations: the feature importance ranking and the heatmap showing how these important features vary across groups.

In [17]:
viz_files = [
    ("Feature importance", "feature_importance.qzv"),
    ("Heatmap", "heatmap.qzv"),
]

for label, fname in viz_files:
    viz_path = f"{qiime_clf_dir}/{fname}"
    print(f"Showing visualization: {label}")
    viz = Visualization.load(viz_path)
    display(viz)

Showing visualization: Feature importance


<visualization: Visualization uuid: 34cf9f8f-d879-4f4f-b3bc-04d7df2ae1d2>

Showing visualization: Heatmap


<visualization: Visualization uuid: 942f2942-7ed1-4b5e-b520-178b20f8e386>

In addition to tabular feature importance, a heatmap is generated to visualize how the most important taxa vary across samples and classes. This heatmap focuses on the top-ranked features and groups samples by the predicted metadata variable, providing a more intuitive view of class-specific abundance patterns.

In [18]:
cmd = f"""
qiime sample-classifier heatmap \
  --i-table "{out_dir}/collapsed_table_filtered.qza" \
  --i-importance "{qiime_clf_dir}/feature_importance.qza" \
  --m-sample-metadata-file "{clean_meta_path}" \
  --m-sample-metadata-column "{column_to_investigate}" \
  --p-group-samples \
  --p-feature-count 30 \
  --o-filtered-table "{qiime_clf_dir}/important-feature-table-top-30.qza" \
  --o-heatmap "{qiime_clf_dir}/important-feature-heatmap.qzv"
"""

print(cmd)
!bash -c "{cmd}"


qiime sample-classifier heatmap   --i-table "../data/processed/classification/results-diet_milk-genus-RandomForestClassifier/collapsed_table_filtered.qza"   --i-importance "../data/processed/classification/qiime-classifier-diet_milk-genus-RandomForestClassifier/feature_importance.qza"   --m-sample-metadata-file "../data/processed/classification/results-diet_milk-genus-RandomForestClassifier/metadata_cleaned_diet_milk.tsv"   --m-sample-metadata-column "diet_milk"   --p-group-samples   --p-feature-count 30   --o-filtered-table "../data/processed/classification/qiime-classifier-diet_milk-genus-RandomForestClassifier/important-feature-table-top-30.qza"   --o-heatmap "../data/processed/classification/qiime-classifier-diet_milk-genus-RandomForestClassifier/important-feature-heatmap.qzv"

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is s

The heatmap  below shows the abundance patterns of the top 30 most important taxa across samples. Grouping samples by the target variable highlights taxa that are enriched or depleted in specific classes.

In [19]:
Visualization.load(f"{qiime_clf_dir}/important-feature-heatmap.qzv")

<visualization: Visualization uuid: 220c5c18-a11a-4d7f-8130-0a4145c627d8>

# References




QIIME 2 Documentation. Sample Classifier Tutorial. Version 2024.5.
https://docs.qiime2.org/2024.5/tutorials/sample-classifier/